# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Data: `data.csv` (dictionary in `columns.csv`), following `checklist.md`.

**Design decisions (made before modelling)**
- No date/time column exists, and every `device_serial` appears exactly once (checked below), so there is no temporal or entity structure to respect: a stratified random split is used for the test set and for CV. Row order is not assumed to be time.
- `device_serial` is an identifier and is dropped. All other columns (channel, device age at sale, price, registered daily use) are known at sale/registration, i.e. at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print(df.claim_within_year.value_counts(normalize=True))

## 1. Hold out the test set (before any exploration or modelling)

In [ ]:
y=df['claim_within_year']; X=df.drop(columns=['claim_within_year','device_serial'])
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te))
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)

## 2. Candidates, fixed before scoring anything
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: one-hot channel + standardised numerics, grid over C in {0.01, 0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, grid over learning_rate {0.03, 0.1} x max_depth {2, 3} (small, early-stopping off, 150 iterations).

Selection rule: highest mean CV ROC AUC (training rows only); the model must beat baseline. Threshold: the one maximising F1 on out-of-fold CV predictions. All preprocessing lives inside the pipelines, so it is refit within each fold.

In [ ]:
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
pre=lambda: ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),('n',StandardScaler(),num)])
cands={
 'baseline':(Pipeline([('m',DummyClassifier(strategy='prior'))]),{}),
 'logreg':(Pipeline([('p',pre()),('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat)],remainder='passthrough')),
                  ('m',HistGradientBoostingClassifier(max_iter=150,early_stopping=False,random_state=SEED))]),
        {'m__learning_rate':[0.03,0.1],'m__max_depth':[2,3]}),
}
fitted={}
for n,(pipe,grid) in cands.items():
    if not grid:
        s=cross_val_predict(pipe,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
        fitted[n]=(pipe,roc_auc_score(y_tr,s))
        print(n,'CV AUC (pooled OOF) %.3f'%fitted[n][1])
    else:
        gs=GridSearchCV(pipe,grid,cv=cv,scoring='roc_auc',refit=True).fit(X_tr,y_tr)
        fitted[n]=(gs.best_estimator_,gs.best_score_)
        print(n,'CV AUC %.3f'%gs.best_score_,gs.best_params_)

In [ ]:
# CV-only model + threshold selection
best=max((n for n in fitted if n!='baseline'),key=lambda n:fitted[n][1])
model=fitted[best][0]
oof=cross_val_predict(model,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
ths=np.linspace(0.02,0.6,59); f1s=[f1_score(y_tr,oof>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('CV AP: %.3f (prevalence %.3f)'%(average_precision_score(y_tr,oof),y_tr.mean()))
print(f'CHOSEN: {best}, threshold={thr:.2f}, CV F1={max(f1s):.3f}, CV AUC={fitted[best][1]:.3f} vs baseline {fitted["baseline"][1]:.3f}')

## 3. Decision (recorded before test scoring)
The model and threshold printed above (selected on training-fold CV only) are final. The test set is scored once below, with no further changes.

## 4. Single test-set evaluation (headline)

In [ ]:
model.fit(X_tr,y_tr)
p=model.predict_proba(X_te)[:,1]; pred=p>=thr
print('Test ROC AUC: %.3f'%roc_auc_score(y_te,p))
print('Test avg precision: %.3f (prevalence %.3f)'%(average_precision_score(y_te,p),y_te.mean()))
print('At threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(thr,precision_score(y_te,pred),recall_score(y_te,pred),f1_score(y_te,pred)))
rng=np.random.default_rng(SEED); yt=y_te.values; b=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt))
    if yt[i].min()!=yt[i].max(): b.append(roc_auc_score(yt[i],p[i]))
print('Bootstrap 95%% CI for AUC: %.3f-%.3f'%tuple(np.percentile(b,[2.5,97.5])))

## Notes
- The test result is the only unbiased estimate; CV numbers above were used for selection.
- Without a date column we cannot assess drift over time; if the model will score future sales, a time-ordered validation should be done once sale dates are available.
- Data is one row per device, so no entity leakage is possible within this file.